# xlingual-drift: Sarvam-30B runner

Runs the RQ1 final test (`configs/rq1_long_sarvam30b.yaml`: 10 shop + 10 pay instances, five
languages, 100 episodes) on **Sarvam-30B** (`sarvamai/sarvam-30b`), an Indic-first open-weight model
(Apache 2.0; Mixture-of-Experts with about 2.4B active parameters), self-hosted with **SGLang**.

Why Sarvam: the Qwen model reasons mostly in English even for Indic requests. A model built for
Indian languages may keep the task state in Hindi or Tamil, which is where drift should show if it
exists.

**Needs an A100 80GB** (the BF16 weights are about 60GB). The first start downloads them.

Serving notes (checked 26 Oct 2026): vLLM support for this model was still an open PR; Sarvam's own
model card serves it with SGLang using the `glm45` tool-call and reasoning parsers, which is what this
notebook does. Cell 6 sends one tool-calling request and stops the notebook if tool calls do not come
back parsed, so no GPU time is spent on a broken setup. Sarvam's hosted API retired `sarvam-30b`
(it points to Sarvam-105B); an optional API cell is at the end.

In [ ]:
# 1. GPU check: Sarvam-30B in BF16 needs an 80GB card.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone or update the repo. The repo is private, so this uses a GitHub token
#    stored in Colab Secrets as GITHUB_TOKEN (key icon in the left sidebar).
import os
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/xlingual-drift"):
    !git clone -q https://{token}@github.com/khalerishabh/xlingual-drift.git /content/xlingual-drift
else:
    !cd /content/xlingual-drift && git pull -q
%cd /content/xlingual-drift
!git log --oneline -1

In [ ]:
# 3. Install the harness (Colab Python) and SGLang (own venv on Python 3.12, so Colab's torch is untouched;
#    Colab moved to Python 3.13, for which SGLang's kernels have no prebuilt wheels).
#    Tries the PyPI wheel first; if it fails or lacks Sarvam support, installs SGLang from source,
#    which is what the Sarvam model card recommends.
import os, yaml, subprocess
!pip install -q -r requirements.txt
!pip install -q uv
CONFIG = "configs/rq1_long_sarvam30b.yaml"
RUN = yaml.safe_load(open(CONFIG))
PY = "/content/sglang-env/bin/python"
CHECK = """
import sglang, torch
print('sglang', sglang.__version__, '| torch', torch.__version__, '| GPU', torch.cuda.is_available())
try:
    from sglang.srt.models.registry import ModelRegistry
    names = [a for a in ModelRegistry.models if 'sarvam' in a.lower()]
    print('sarvam architectures:', names or 'NONE')
except Exception as e:
    print('registry check skipped:', type(e).__name__)
"""
def sglang_status():
    if not os.path.exists(PY):
        return "missing"
    r = subprocess.run([PY, "-c", CHECK], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-1500:])
    if r.returncode != 0:
        return "broken"
    return "no_sarvam" if "NONE" in r.stdout else "ok"
if not os.path.exists(PY):
    !uv venv -q --python 3.12 /content/sglang-env
if sglang_status() != "ok":
    !uv pip install -p {PY} sglang --torch-backend=auto 2>&1 | tail -n 25
if sglang_status() != "ok":
    !rm -rf /content/sglang && git clone -q --depth 1 https://github.com/sgl-project/sglang.git /content/sglang
    !uv pip install -p {PY} -e /content/sglang/python --torch-backend=auto 2>&1 | tail -n 25
status = sglang_status()
if status in ("missing", "broken"):
    raise SystemExit("SGLang did not install. Send the output of this cell.")
if status == "no_sarvam":
    print("Warning: no Sarvam architecture registered; cell 6 will show whether the model still serves.")


In [ ]:
# 4. Raw logs go to Drive so a disconnect never loses finished episodes.
from google.colab import drive
drive.mount("/content/drive")
RUNS_DIR = "/content/drive/MyDrive/xlingual-drift-runs"
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# 5. Serve Sarvam-30B with SGLang and record exactly what was served.
import json, subprocess, time, urllib.request
from huggingface_hub import HfApi
serving = RUN["serving"]
gpu_name, vram_mib = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split(",")]
vram_gb = int(vram_mib) / 1024
if vram_gb < serving["min_vram_gb"]:
    raise SystemExit(f"Sarvam-30B needs >= {serving['min_vram_gb']} GB; this runtime has {gpu_name} ({vram_gb:.0f} GB). Switch to an A100 80GB.")
sha = HfApi().model_info(serving["model"], revision=serving["revision"]).sha
engine = subprocess.run(["/content/sglang-env/bin/python", "-c", "import sglang; print(sglang.__version__)"],
                        capture_output=True, text=True).stdout.strip()
SERVING_JSON = "/content/serving.json"
json.dump({"model": serving["model"], "revision": sha, "vllm_version": f"sglang-{engine}", "gpu_name": gpu_name,
           "vram_gb": round(vram_gb), "max_model_len": serving["max_model_len"]}, open(SERVING_JSON, "w"), indent=2)
print(open(SERVING_JSON).read())
cmd = ["/content/sglang-env/bin/python", "-m", "sglang.launch_server",
       "--model-path", serving["model"], "--revision", sha, "--trust-remote-code",
       "--served-model-name", serving["model"], "--port", "8000",
       "--context-length", str(serving["max_model_len"]), "--mem-fraction-static", "0.88",
       "--tool-call-parser", serving["tool_call_parser"], "--reasoning-parser", serving["reasoning_parser"],
       *serving.get("extra_args", [])]
env = {**os.environ, "PATH": "/content/sglang-env/bin:" + os.environ["PATH"]}
server = subprocess.Popen(cmd, stdout=open("/content/sglang.log", "w"), stderr=subprocess.STDOUT, env=env)
for i in range(240):
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=5).status == 200:
            print(f"server up after ~{i * 10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        raise RuntimeError("SGLang exited; see /content/sglang.log (next cell)")
    time.sleep(10)
else:
    raise TimeoutError("server not up after 40 min")

In [ ]:
# If the server failed to start, the reason is here.
!tail -n 40 /content/sglang.log

In [ ]:
# 6. One tool-calling request. Stops here if the tool call does not come back parsed.
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
tools = [{"type": "function", "function": {"name": "get_balance", "description": "Balance of an account.",
          "parameters": {"type": "object", "properties": {"account_id": {"type": "string"}}, "required": ["account_id"]}}}]
resp = client.chat.completions.create(model=RUN["cells"][0]["model"], tools=tools, tool_choice="auto", max_tokens=2048,
    messages=[{"role": "user", "content": "Mere savings account ACC-202 ka balance check karo."}])
msg = resp.choices[0].message
print("tool_calls:", msg.tool_calls)
print("content:", (msg.content or "")[:300])
print("reasoning:", (getattr(msg, "reasoning_content", None) or "")[:300])
if not msg.tool_calls:
    raise SystemExit("No parsed tool call: check the parser flags in configs/rq1_long_sarvam30b.yaml against the model card before running.")

In [ ]:
# 7. Harness sanity, then the run. --resume continues across sessions.
!python -m pytest -q
LOG = f"{RUNS_DIR}/{RUN['run_name']}.jsonl"
print("log:", LOG)
!python -m eval.run_grid --config {CONFIG} --serving {SERVING_JSON} --resume --workers 16 --out {LOG}

In [ ]:
# 8. Drift report with the pre-registered RQ1 decision (scored on the end state, see eval/rescore_long.py).
from eval.drift import load, report
RESCORED = LOG.replace(".jsonl", ".rescored.jsonl")
!python -m eval.rescore_long {LOG} {RESCORED}
print(report(load(RESCORED)))

In [ ]:
# 9. Download the raw log.
from google.colab import files
files.download(LOG)

## Optional: Sarvam hosted API instead of self-hosting

Only if self-hosting is not possible. The hosted API retired `sarvam-30b`, so this runs whichever
model id you set below (check the ids enabled for your key in the Sarvam dashboard). API models cannot
be pinned to exact weights, so these episodes carry no backbone fingerprint and go to a separate log.
Store the key in Colab Secrets as `SARVAM_API_KEY`.

In [ ]:
# 10. API run (optional). Writes a config variant at run time; nothing in the repo changes.
from google.colab import userdata
os.environ["SARVAM_API_KEY"] = userdata.get("SARVAM_API_KEY")
SARVAM_API_MODEL = "sarvam-105b"
api = yaml.safe_load(open(CONFIG))
api["run_name"] = f"rq1_long_sarvam_api_{SARVAM_API_MODEL}"
api.pop("serving")
api["cells"] = [{"name": f"sarvam_api_{SARVAM_API_MODEL}", "policy": "openai_compat", "model": SARVAM_API_MODEL,
                 "base_url": "https://api.sarvam.ai/v1", "api_key_env": "SARVAM_API_KEY",
                 "api_key_header": "api-subscription-key", "temperature": 0.6, "max_tokens": 8192, "seeds": [1]}]
yaml.safe_dump(api, open("/content/sarvam_api.yaml", "w"))
API_LOG = f"{RUNS_DIR}/{api['run_name']}.jsonl"
!python -m eval.run_grid --config /content/sarvam_api.yaml --resume --workers 4 --out {API_LOG}
from eval.drift import load, report
print(report(load(API_LOG)))